# Iris classification with scikit-learn

Here we use the well-known Iris species dataset to illustrate how SHAP can explain the output of many different model types, from k-nearest neighbors, to neural networks. This dataset is very small, with only a 150 samples. We use a random set of 130 for training and 20 for testing the models. Because this is a small dataset with only a few features we use the entire training dataset for the background. In problems with more features we would want to pass only the median of the training dataset, or weighted k-medians. While we only have a few samples, the prediction problem is fairly easy and all methods acheive perfect accuracy. What's interesting is how different methods sometimes rely on different sets of features for their predictions.

## Load the data

In [ ]:
import time

import numpy as np
import sklearn
from sklearn.model_selection import train_test_split

import shap

X, y = shap.datasets.iris()
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# rather than use the whole training set to estimate expected values, we could summarize with
# a set of weighted kmeans, each weighted by the number of points they represent. But this dataset
# is so small we don't worry about it
# X_train_summary = shap.kmeans(X_train, 50)


def print_accuracy(f):
    print(f"Accuracy = {100 * np.sum(f(X_test) == Y_test) / len(Y_test)}%")
    time.sleep(0.5)  # to let the print get out before any progress bars


shap.initjs()

## K-nearest neighbors

In [ ]:
knn = sklearn.neighbors.KNeighborsClassifier()
knn.fit(X_train, Y_train)

print_accuracy(knn.predict)

### Explain a single prediction from the test set

In [ ]:
explainer = shap.Explainer(knn.predict_proba, X_train)
shap_values = explainer(X_test)

# visualize the first prediction's explanation for the 'setosa' class
shap.plots.force(shap_values[0, ..., 0])

### Explain all the predictions in the test set

In [ ]:
# visualize all the predictions' explanations for the 'setosa' class
shap.plots.force(shap_values[..., 0])

## Support vector machine with a linear kernel

In [ ]:
svc_linear = sklearn.svm.SVC(kernel="linear", probability=True)
svc_linear.fit(X_train, Y_train)
print_accuracy(svc_linear.predict)

# explain all the predictions in the test set
explainer = shap.Explainer(svc_linear.predict_proba, X_train)
shap_values = explainer(X_test)

# visualize the training set predictions
shap.plots.force(shap_values[..., 0])

## Support vector machine with a radial basis function kernel

In [ ]:
svc_rbf = sklearn.svm.SVC(kernel="rbf", probability=True)
svc_rbf.fit(X_train, Y_train)
print_accuracy(svc_rbf.predict)

# explain all the predictions in the test set
explainer = shap.Explainer(svc_rbf.predict_proba, X_train)
shap_values = explainer(X_test)

# visualize the training set predictions
shap.plots.force(shap_values[..., 0])

## Logistic regression

In [ ]:
linear_lr = sklearn.linear_model.LogisticRegression(solver="newton-cg")
linear_lr.fit(X_train, Y_train)
print_accuracy(linear_lr.predict)

# explain all the predictions in the test set
explainer = shap.Explainer(linear_lr.predict_proba, X_train)
shap_values = explainer(X_test)

# visualize the training set predictions
shap.plots.force(shap_values[..., 0])

## Decision tree

In [ ]:
import sklearn.tree

dtree = sklearn.tree.DecisionTreeClassifier(min_samples_split=2)
dtree.fit(X_train, Y_train)
print_accuracy(dtree.predict)

# explain all the predictions in the test set
explainer = shap.Explainer(dtree.predict_proba, X_train)
shap_values = explainer(X_test)

# visualize the training set predictions
shap.plots.force(shap_values[..., 0])

## Random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rforest = RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_split=2, random_state=0)
rforest.fit(X_train, Y_train)
print_accuracy(rforest.predict)

# explain all the predictions in the test set
explainer = shap.Explainer(rforest.predict_proba, X_train)
shap_values = explainer(X_test)

# visualize the training set predictions
shap.plots.force(shap_values[..., 0])

## Neural network

In [ ]:
from sklearn.neural_network import MLPClassifier

nn = MLPClassifier(solver="lbfgs", alpha=1e-1, hidden_layer_sizes=(5, 2), random_state=0)
nn.fit(X_train, Y_train)
print_accuracy(nn.predict)

# explain all the predictions in the test set
explainer = shap.Explainer(nn.predict_proba, X_train)
shap_values = explainer(X_test)

# visualize the training set predictions
shap.plots.force(shap_values[..., 0])

## Summary Plot

While force plots are great for seeing how each feature contributes to a single prediction, a beeswarm plot can show the relative importance of features across the entire test set. Here we show the beeswarm plot for the 'setosa' species using the Random Forest model.

In [ ]:
explainer = shap.Explainer(rforest.predict_proba, X_train)
shap_values = explainer(X_test)

# summarize the effects of all the features
shap.plots.beeswarm(shap_values[..., 0])